# Hubble-8B - single-example showcase (logit lens + LSR, forget vs retain)

Illustrative single example for a slide; the aggregate (master) is the real claim.

**Example selection is baseline-clean and method-agnostic:** among entities Perturbed knows best (lowest birthdate rank), pick the one whose occupation-LSR most cleanly separates Perturbed from Standard. We never select on the unlearning methods' outcomes.

**LSR here accounts for both forget and retain:** per-set calibration `(LSR - Standard) / (Perturbed - Standard)` (0 = never-knew, 1 = knows), summarized by `LSR_selectivity = retain_cal - forget_cal` (1 = faithful selective erasure).

In [ ]:
# --- Setup: install deps, put hubble8b on path, set HF token ---
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.53", "datasets", "accelerate", "bitsandbytes", "scipy"], check=False)
HUB_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
for cand in [HUB_DIR, os.path.join(HUB_DIR, "gpu_8b_run"), "/kaggle/input"]:
    if os.path.isdir(os.path.join(cand, "hubble8b")):
        sys.path.insert(0, cand); break
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t: os.environ["HF_TOKEN"] = t
    except Exception:
        pass
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF_TOKEN is not set. Export HF_TOKEN before submitting the job.")
import hubble8b as h8
h8.hf_login()
print("device:", h8.DEVICE, "| target slot:", h8.TARGET_SLOT, "| LSR slot:", h8.LSR_SLOT)

In [ ]:
# --- Data: facts for both slots on both sets ---
import numpy as np, json, torch
forget_set, keep_set, unseen_set = h8.load_yago_splits()
tok = h8.get_tokenizer()
f_bd = h8.facts_for(forget_set, h8.TARGET_SLOT); k_bd = h8.facts_for(keep_set, h8.TARGET_SLOT)  # birthdate (rank)
f_oc = h8.facts_for(forget_set, h8.LSR_SLOT);    k_oc = h8.facts_for(keep_set, h8.LSR_SLOT)      # occupation (LSR)
print(f"forget entities: {len(f_bd)} | retain entities: {len(k_bd)}")

## Select the baseline-clean forget & retain example

In [ ]:
# --- Baseline-clean example selection (method-agnostic; also conditions LSR calibration) ---
# Criterion: among the entities Perturbed KNOWS best (lowest birthdate gold rank), pick the one
# whose occupation-LSR most cleanly separates Perturbed (knows) from Standard (never knew).
# Uses ONLY the two baselines -> never selects on the unlearning methods' outcomes (no cherry-pick).
DEVICE = h8.DEVICE
def final_rank(m, prefix, gold):
    return h8.layerwise_target_rank(m, tok, prefix, gold, DEVICE)[-1]
def lsr_of(m, prefix, gold, E):
    r = h8.layerwise_residual(m, tok, prefix, gold, E, DEVICE)
    return h8.lsr_scalar(r[0]) if r else float("nan")

pert = h8.load_named("Perturbed"); E_REF = h8.build_Eref(pert).to(DEVICE)
P = dict(f_rank=[], f_lsr=[], k_rank=[], k_lsr=[])
for (pf, pg), (of, og) in zip(f_bd, f_oc):
    P["f_rank"].append(final_rank(pert, pf, pg)); P["f_lsr"].append(lsr_of(pert, of, og, E_REF))
for (pf, pg), (of, og) in zip(k_bd, k_oc):
    P["k_rank"].append(final_rank(pert, pf, pg)); P["k_lsr"].append(lsr_of(pert, of, og, E_REF))
h8.to_cpu(pert); del pert; h8.free_cuda()

std = h8.load_named("Standard")
S = dict(f_lsr=[lsr_of(std, of, og, E_REF) for of, og in f_oc],
         k_lsr=[lsr_of(std, of, og, E_REF) for of, og in k_oc])
h8.to_cpu(std); del std; h8.free_cuda()

def pick(rank, p_lsr, s_lsr, topn=15):
    known = list(np.argsort(rank)[:topn])                      # Perturbed knows the birthdate best
    return int(max(known, key=lambda i: p_lsr[i] - s_lsr[i]))  # + cleanest LSR discrimination
# Allow manual override: set FORGET_IDX / KEEP_IDX before running to feature a specific entity.
FORGET_IDX = int(pick(P["f_rank"], P["f_lsr"], S["f_lsr"]))
KEEP_IDX   = int(pick(P["k_rank"], P["k_lsr"], S["k_lsr"]))
f_prefix, f_gold = f_bd[FORGET_IDX]; fo_prefix, fo_gold = f_oc[FORGET_IDX]
k_prefix, k_gold = k_bd[KEEP_IDX];   ko_prefix, ko_gold = k_oc[KEEP_IDX]
f_name = json.loads(forget_set[FORGET_IDX]["meta"])["full_name"]
k_name = json.loads(keep_set[KEEP_IDX]["meta"])["full_name"]
print(f"FORGET [{FORGET_IDX}] {f_name}: Perturbed rank={P['f_rank'][FORGET_IDX]:.0f} "
      f"LSR(P={P['f_lsr'][FORGET_IDX]:.2f} S={S['f_lsr'][FORGET_IDX]:.2f}) | {f_gold!r} / occ {fo_gold!r}")
print(f"RETAIN [{KEEP_IDX}] {k_name}: Perturbed rank={P['k_rank'][KEEP_IDX]:.0f} "
      f"LSR(P={P['k_lsr'][KEEP_IDX]:.2f} S={S['k_lsr'][KEEP_IDX]:.2f}) | {k_gold!r} / occ {ko_gold!r}")

## Compute rank / LSR / top-k for the chosen example

In [ ]:
# --- Compute rank / LSR / top-k for every model on the chosen example ---
ALL = ["Standard", "Perturbed", "NPO", "GradDiff", "IDK", "RMU"]
rank, lsr, topk = {}, {}, {}
def topk_at(m, prefix, k=5):
    ids = tok(prefix, return_tensors="pt").input_ids.to(DEVICE)
    with torch.no_grad():
        lg = m(input_ids=ids).logits[0, -1].float()
    p = torch.softmax(lg, -1); t = torch.topk(p, k)
    toks = tok.convert_ids_to_tokens(t.indices.tolist())          # raw subwords (no empty strips)
    return [(tk, float(v)) for tk, v in zip(toks, t.values.tolist())]
for name in ALL:
    try:
        m = h8.load_named(name)
    except Exception as e:
        print(f"  {name}: unavailable ({type(e).__name__})"); continue
    rank[name] = dict(forget=h8.layerwise_target_rank(m, tok, f_prefix, f_gold, DEVICE),
                      retain=h8.layerwise_target_rank(m, tok, k_prefix, k_gold, DEVICE))
    rf = h8.layerwise_residual(m, tok, fo_prefix, fo_gold, E_REF, DEVICE)
    rk = h8.layerwise_residual(m, tok, ko_prefix, ko_gold, E_REF, DEVICE)
    lsr[name]  = dict(forget=(rf[0] if rf else None), retain=(rk[0] if rk else None))
    topk[name] = topk_at(m, f_prefix)
    print(f"  [{name}] forget rank={rank[name]['forget'][-1]:.0f} retain={rank[name]['retain'][-1]:.0f}")
    h8.to_cpu(m); del m; h8.free_cuda()

## Figure 1 - rank-by-layer logit lens (forget vs retain)

In [ ]:
# --- Figure 1: rank-by-layer, 4 methods x [forget|retain] vs Standard/Perturbed refs ---
import matplotlib.pyplot as plt
METHODS = [m for m in ["NPO", "GradDiff", "IDK", "RMU"] if m in rank]
REF = [("Standard", dict(ls="--", c="#888")), ("Perturbed", dict(ls="--", c="#111"))]
fig, axes = plt.subplots(len(METHODS), 2, figsize=(13, 3.1*len(METHODS)), sharex=True, squeeze=False)
for r, meth in enumerate(METHODS):
    for c, split in enumerate(["forget", "retain"]):
        ax = axes[r, c]
        for ref, st in REF:
            if ref in rank:
                cur = rank[ref][split]; ax.plot(range(len(cur)), cur, label=ref, lw=1.5, **st)
        cur = rank[meth][split]
        ax.plot(range(len(cur)), cur, label=meth, lw=2.5, color=h8.COLOR_MAP[meth])
        ax.set_yscale("log"); ax.grid(alpha=0.3)
        if r == 0: ax.set_title("Forget (Unlearn)" if split == "forget" else "Retain (Keep)", fontweight="bold")
        if c == 0: ax.set_ylabel(f"{meth}\ngold rank (log)")
        if r == 0 and c == 1: ax.legend(fontsize=8, loc="upper left")
for ax in axes[-1]: ax.set_xlabel("hidden-state index (0 = embeddings .. N = final)")
fig.suptitle(f"Rank-by-layer logit lens - single example\nforget: {f_name} ({f_gold!r})   |   retain: {k_name} ({k_gold!r})",
             fontweight="bold")
fig.tight_layout()
p = f"{h8.workdir()}/showcase_rank_8b.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); h8.upload_artifact(p); plt.show()

## Figure 2 - LSR by layer (raw + calibrated)

In [ ]:
# --- Figure 2: LSR (C_f mass) by layer, same layout; raw + calibrated scalar annotated ---
import matplotlib.pyplot as plt
def _sc(name, split):
    c = lsr.get(name, {}).get(split); return h8.lsr_scalar(c) if c is not None else float("nan")
Sf, Pf = _sc("Standard", "forget"), _sc("Perturbed", "forget")
Sk, Pk = _sc("Standard", "retain"), _sc("Perturbed", "retain")
def calib(v, s, p):
    d = p - s
    return (v - s) / d if abs(d) > 1e-6 else float("nan")   # 0 = never-knew level, 1 = knows level
fig, axes = plt.subplots(len(METHODS), 2, figsize=(13, 3.1*len(METHODS)), sharex=True, squeeze=False)
for r, meth in enumerate(METHODS):
    for c, split in enumerate(["forget", "retain"]):
        ax = axes[r, c]
        for ref, st in REF:
            cur = lsr.get(ref, {}).get(split)
            if cur is not None: ax.plot(range(len(cur)), cur, label=ref, lw=1.5, **st)
        cur = lsr[meth][split]
        if cur is not None:
            ax.plot(range(len(cur)), cur, label=meth, lw=2.5, color=h8.COLOR_MAP[meth])
            raw = h8.lsr_scalar(cur)
            cal = calib(raw, Sf, Pf) if split == "forget" else calib(raw, Sk, Pk)
            ax.text(0.98, 0.9, f"LSR={raw:.2f}  (cal {cal:+.2f})", transform=ax.transAxes,
                    ha="right", va="top", fontsize=8, color=h8.COLOR_MAP[meth])
        ax.set_ylim(0, 1); ax.grid(alpha=0.3)
        if r == 0: ax.set_title(("Forget" if split == "forget" else "Retain") + " - occupation", fontweight="bold")
        if c == 0: ax.set_ylabel(f"{meth}\nC_f mass")
        if r == 0 and c == 0: ax.legend(fontsize=8, loc="upper left")
for ax in axes[-1]: ax.set_xlabel("hidden-state index")
fig.suptitle(f"LSR (semantic residual) by layer - single example\nforget occ: {fo_gold!r}   |   retain occ: {ko_gold!r}\n"
             f"calibrated: 0 = never-knew (Standard), 1 = knows (Perturbed)", fontweight="bold")
fig.tight_layout()
p = f"{h8.workdir()}/showcase_lsr_8b.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); h8.upload_artifact(p); plt.show()

## Calibrated LSR selectivity + top-k + summary

In [ ]:
# --- Calibrated LSR (forget+retain) selectivity, top-k, summary, prompt box ---
import pandas as pd, matplotlib.pyplot as plt
rows = []
for name in ["Standard", "Perturbed", "NPO", "GradDiff", "IDK", "RMU"]:
    if name not in rank: continue
    fl, rl = _sc(name, "forget"), _sc(name, "retain")
    fcal = calib(fl, Sf, Pf); rcal = calib(rl, Sk, Pk)
    rows.append(dict(model=name,
                     forget_rank=round(rank[name]["forget"][-1], 1),
                     retain_rank=round(rank[name]["retain"][-1], 1),
                     forget_LSR=round(fl, 3), retain_LSR=round(rl, 3),
                     forget_cal=round(fcal, 2), retain_cal=round(rcal, 2),
                     LSR_selectivity=round(rcal - fcal, 2),   # ~1 faithful selective, ~0 indiscriminate
                     forget_top1=topk[name][0][0] if name in topk else ""))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
sp = f"{h8.workdir()}/showcase_summary.csv"; summary.to_csv(sp, index=False); h8.upload_artifact(sp)

# LSR selectivity bar (methods only) = retain_cal - forget_cal
meth_rows = summary[summary.model.isin(["NPO", "GradDiff", "IDK", "RMU"])]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(meth_rows.model, meth_rows.LSR_selectivity, color=[h8.COLOR_MAP[m] for m in meth_rows.model])
ax.axhline(0, c="k", lw=0.8); ax.set_ylabel("LSR selectivity  (retain_cal - forget_cal)")
ax.set_title("LSR selectivity - erase forget concept, keep retain concept\n(1 = faithful selective, 0 = indiscriminate)")
ax.grid(axis="y", alpha=0.3)
p = f"{h8.workdir()}/showcase_lsr_selectivity_8b.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); h8.upload_artifact(p); plt.show()

# top-k table
tk = pd.DataFrame([dict(model=n, top5=", ".join(f"{t}({p:.2f})" for t, p in topk[n])) for n in topk])
print("\nTop-5 predicted tokens for the forget gold slot ('what it says instead'):")
print(tk.to_string(index=False))
tp = f"{h8.workdir()}/showcase_topk.csv"; tk.to_csv(tp, index=False); h8.upload_artifact(tp)

# annotated prompt box (monospace, slide-ready)
box = (f"FORGET  |  {f_name}\n   prompt: {f_prefix!r}\n   gold birthdate: {f_gold!r}    occupation: {fo_gold!r}\n\n"
       f"RETAIN  |  {k_name}\n   prompt: {k_prefix!r}\n   gold birthdate: {k_gold!r}    occupation: {ko_gold!r}\n\n"
       + summary.to_string(index=False))
fig, ax = plt.subplots(figsize=(13, 2.4 + 0.3*len(summary))); ax.axis("off")
ax.text(0.0, 1.0, box, va="top", ha="left", family="monospace", fontsize=9)
p = f"{h8.workdir()}/showcase_prompts.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); h8.upload_artifact(p); plt.show()